In [1]:
!pip install -q\
transformers==4.40.0 \
accelerate==0.29.3 \
peft==0.10.0 \
trl==0.9.6 \
bitsandbytes

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig,PeftModel
from trl import SFTTrainer

ORIGINAL DATASET-

In [3]:
model_name='NousResearch/Llama-2-7b-chat-hf'
dataset_name="mlabonne/guanaco-llama2-1k"
new_model='Llama-2-7b-chat-finetune'

#QLoRA parameters

#LoRA attention dimension,r for rank
lora_r=64
#Alpha parameter for Lora Scaling
lpra_alpha=16
#Dropout probablity for Lora layers
lora_dropout=0.1
#bitsandbytesparameter


#activate dtype for 4-bit base models
use_4bit=True
bnb_4bit_compute_dtype='float16'
#Quatntisation type (fp4 or nf4)
bnb_4bit_quant_type='nf4'
#actvate nested quantisation for 4 bit base models (double quantisation
use_nested_quant=False

#Training Argument parameters
#output directory where the model predictions will be stored
output_dir="./results"
#no. of training epochs
num_train_epochs=1
#enable fp16/bf16 training(set bf16 to True with an A100)
fp16=False
bf16=False

#Batch size per GPU for training
per_device_train_batch_size=1
#Batch size per GPU for evaluation
per_device_eval_batch_size=4
#Number of update steps to accumulate the gradients for
gradient_accumulation_steps=4
#Enable gradient checkpointing
gradient_checkpointing=False
#Maximum gradient Normal(gradient clipping)
max_grad_norm=0.3
#Initial learning rate (AdamW optimizer)
learning_rate=2e-4
#weight decay to apply to all layers except bias/LayerNorm weights
weight_decay=0.001
#optimiser to use
optim="paged_adamw_8bit"
#learning rate schedule
lr_scheduler_type='cosine'
#Number of training steps (overrides num_train_epochs)
max_steps=-1
#ratio of steps for a linear warmup(from 0 to learning rate)
warmup_ratio=0.03
#group sequences into batches with same length
#saves memory and speed up training considerably
group_by_length=True

#save checkponit every x update steps
save_steps=0
#Log every X UPDATE STPES
logging_steps=25


#SFT parameter

#maximum sequenc length to use
max_seq_length=None
#packing multiple short examples in the same input sentence to increase efficiency
packing=False

#Load the entire model on thr GpU 0
device_map = {"": 0}


In [4]:
#Load dataset
from torch.optim import AdamW
dataset=load_dataset(dataset_name,split="train")
#load tokeniser and model with QLORA configuration
compute_dtype=torch.float16
bnb_config=BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
    )
 #check gpu compatibility with bfloat16
if compute_dtype==torch.float16 and use_4bit:
  major, _ = torch.cuda.get_device_capability()
  if major >= 8:
    print('=',*80)
    print('Your GPU supports bfloat16: accelerate training with bf16=True')
    print('=',*80)

# Load base model
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
    torch_dtype=torch.float16
)
model.config.use_cache=False
model.config.pretraining_tp=1
#load LlaMA tokenizer
tokenizer=AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'#fix weird overflow issue with fp16 training

#Load LoRA configuration
peft_config=LoraConfig(
    lora_alpha=lpra_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias='none',
    task_type='CAUSAL_LM'
)
#set training parameters
training_arguments=TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit", # Explicitly use standard AdamW optimizer
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="none"
)
#set supervised finetuning parameters
from trl import SFTTrainer, SFTConfig

# Remove these args from TrainingArguments and put in SFTConfig instead
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=SFTConfig(
        output_dir=output_dir,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",   # ← fix here
        save_steps=save_steps,
        logging_steps=logging_steps,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        fp16=fp16,
        bf16=bf16,
        max_grad_norm=max_grad_norm,
        max_steps=max_steps,
        warmup_ratio=warmup_ratio,
        group_by_length=group_by_length,
        lr_scheduler_type=lr_scheduler_type,
        report_to="none",
        # SFT-specific args go here now
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        packing=packing,
    )
)
optimizer = AdamW(
    trainer.model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)
trainer.optimizer = optimizer


trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:289: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,1.449700
50,1.681700
75,1.210900
100,1.442600
125,1.161400
150,1.358500
175,1.165400
200,1.453900
225,1.156900
250,1.533700


TrainOutput(global_step=250, training_loss=1.3614545135498046, metrics={'train_runtime': 2113.8017, 'train_samples_per_second': 0.473, 'train_steps_per_second': 0.118, 'total_flos': 1.8404903613333504e+16, 'train_loss': 1.3614545135498046, 'epoch': 1.0})

In [5]:
#save trained model
trainer.model.save_pretrained(new_model)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
logging.set_verbosity(logging.CRITICAL)
#run text generation pipeline with our next model
prompt="What is a machine learning?"
pipe=pipeline(task="text-generation",model=model,tokenizer=tokenizer,max_length=300)
result=pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] What is a machine learning? [/INST] Machine learning is a subfield of artificial intelligence that involves training algorithms to learn from data. It allows a computer to learn from experience and improve its performance on a task without being explicitly programmed.

Machine learning algorithms can be used for a wide range of tasks, such as image recognition, natural language processing, and predictive modeling. They are often used in applications such as self-driving cars, medical diagnosis, and financial forecasting.

There are several types of machine learning, including:

1. Supervised learning: In this type of machine learning, the algorithm is trained on a labeled dataset, where the correct output is provided for each input. The algorithm learns to predict the correct output for new inputs.
2. Unsupervised learning: In this type of machine learning, the algorithm is trained on an unlabeled dataset, where no correct output is provided for any input. The algorithm learn